In [ ]:
# run this notebook after 25_missense_lof_R
# use this notebook to generate ancestry clusters, compare dnm counts and spectra by ancestry
# we need ukb sibling and trio dnm counts alongside parental ages (./ukb/ukb_sibs_dnms_parental_ages.tsv, 
# ./ukb/ukb_trios_dnms_parental_ages.tsv)
# we need ukb dnms split by 96-mutation types (./ukb/ukb_96_type_dnms_feb11.csv)
# this notebook is particularly repetitive, as it includes cells that were originally across multiple notebooks
# after this, run 27_methods_smoking_R

In [ ]:
library(jcolors)
library(data.table)
library(ggplot2)
library(dplyr)
library(jsonlite)
library(tidyverse)
library(ggpubr)
library(cluster)
library(xtable)
library(patchwork)
library(ggsignif)

# ── palette & mappings ────────────────────────────────────────────────────────
ancestry_colors = c(
  "EAS"  = "#E69F00", "AFR"  = "#56B4E9", "MID"  = "#009E73",
  "AMR1" = "#F0E442", "AMR2" = "#D55E00","EUR1" = "#0072B2", 
  "EUR2" = "#7B4173", "EUR3" = "#A6761D","SAS"  = "#CC79A7", 
  "OCE"  = "#C0392B"
)

cluster_to_anc = c(
  "1" = "EAS",  "2" = "AFR",  "3" = "MID",  "4" = "AMR1",
  "5" = "EUR1", "6" = "AMR2", "7" = "SAS",  "8" = "EUR2",
  "9" = "EUR3", "10" = "OCE"
)

order = c(9,5,8,6,3,7,4,1,10,2)

legend_labels = c(
  "EAS"  = "EAS",
  "AFR"  = "AFR",
  "MID"  = "MID",
  "AMR1" = "AMR1~PEL-MXL",
  "AMR2" = "AMR2~PUR-CLM",
  "EUR1" = "EUR1~IBS-TSI",
  "EUR2" = "EUR2~FIN",
  "EUR3" = "EUR3~CEU-GBR",
  "OCE"  = "OCE",
  "SAS"  = "SAS"
)


base_theme = theme_classic(base_size = 16) +
  theme(
    axis.text.x        = element_text(angle = 45, hjust = 1, size = 14),
    axis.text.y        = element_text(size = 14),
    axis.title         = element_text(size = 15),
    axis.title.y       = element_text(size = 15, margin = margin(r = 8)),
    panel.grid.major.y = element_line(color = "grey88", linewidth = 0.4)
  )


In [ ]:
set.seed(444)

# read in hgdp 1 kg pcs 
meta = fread("./pca/hgdp_1kg_pcs_biallelic_jan25.txt", header=TRUE)

# elbow plot to determine how many pcs to use 
eigenvalues = fread("./pca/hgdp_1000g_eigenvalues_acaf_biallelic.tsv", header=TRUE)
eigenvalues$pcs =1:20
names(eigenvalues) = c("eigenvalue", "pc")
eigenvalues$var_explained = eigenvalues$eigenvalue/sum(eigenvalues$eigenvalue)
ggplot(data = eigenvalues, aes(x = pc, y = var_explained)) + geom_line() + 
scale_x_continuous(breaks = c(1:20)) + labs(x = "PC", y = "Proportion of variance explained") + theme_bw()

In [ ]:
# k-means clustering

meta_kmeans = meta %>% select(paste("pc", 1:6, sep = ""))
silhouette_method = function(pc, max_k) {
  silhouette_scores = numeric(max_k)
  for (i in 2:max_k) {
    kmeans_model = kmeans(pc, centers = i, nstart = 1000)
    silhouette_obj = silhouette(kmeans_model$cluster, dist(pc))
    silhouette_scores[i] = mean(silhouette_obj[, "sil_width"])
  }
    return(silhouette_scores)
}


# call silhouette_method function
scores = silhouette_method(meta_kmeans, max_k = 15)
plot_scores = data.frame(k = 2:15, score = scores[2:15])
ggplot(data = plot_scores, aes(x = k, y = score)) + geom_point(fill = "black",size=3) + 
theme_bw(base_size = 12) + labs(x = "K", y = "Silhouette score") + 
scale_y_continuous(breaks = seq(from = 0.55, to= 0.75, by = 0.05),
                  limits = c(0.55, 0.75)) + scale_x_continuous(limits = c(2,15), breaks = seq(2, 15, by = 1))
kmeans_result = kmeans(meta_kmeans, centers = 10, nstart = 1000)
meta$cluster = kmeans_result$cluster
for (i in 1:10){
    sample_list = meta$s[which(meta$cluster == i)]
    fwrite(list(sample_list), paste("./pca/cluster_", i, "_samples_biallelic.txt", sep = ""), sep = "\t", row.names=FALSE,
           quote=FALSE, col.names=FALSE)
    
}

In [ ]:
# generate 7 mutation type dataframes for siblings + trios 
trio_difs = fread("./trios_snps_after_standard_QC_GIAB_COSMIC.txt")
cosmic_96 = unique(trio_difs$Type)
cosmic_types_pyr = c(unique(substr(cosmic_96,3, 5)), "CpG>TpG")
trio_difs$pyr = ifelse(substr(trio_difs$Type, 3,7) == "C>T]G", "CpG>TpG", 
                      substr(trio_difs$Type, 3,5))

sib_difs = fread("./sibs_snps_QC_GIAB_distance_AF_outliers_COSMIC.txt")
sib_difs$pyr = ifelse(substr(sib_difs$Type, 3,7) == "C>T]G", "CpG>TpG", 
                      substr(sib_difs$Type, 3,5))
sib_difs$pair = paste(sib_difs$idv1, sib_difs$idv2, sep = "_")

sibs_counts = fread("./relatedness_dec20/passing_sibs_counts_after_qc.txt")
sibs_counts$pair = paste(sibs_counts$idv1, sibs_counts$idv2, sep = "_")
sibs_ibd2 = fread("./aou_ibd2_giab_removed.tsv")
sibs_counts = merge(sibs_counts, sibs_ibd2, by = "pair")
sibs_nums = data.frame(idv = sibs_counts$idv1, pair = sibs_counts$pair)
sibs_nums$count = sibs_counts$idv1_count
sibs_nums$denom = sibs_counts$length_filter
sibs_nums2 = data.frame(idv = sibs_counts$idv2, pair = sibs_counts$pair)
sibs_nums2$count = sibs_counts$idv2_count
sibs_nums2$denom = sibs_counts$length_filter
sibs_nums = rbind(sibs_nums, sibs_nums2)

sibs_parents = fread("./relatedness_dec20/sibs_passing_qc_with_parental_ages.txt") %>% select(off, mother_age, father_age)
sibs_counts_with_ages = merge(sibs_nums, sibs_parents, by.x = "idv", by.y = "off", all.x = TRUE)
sibs_counts_with_ages$source = "sib"
sibs_counts_with_ages$denom = sibs_counts_with_ages$denom*2

# add 7-type counts
sibs_counts_with_ages[cosmic_types_pyr] = NA
for (i in 1:nrow(sibs_counts_with_ages)){
    sib_sub = sib_difs %>% filter(idv == sibs_counts_with_ages$idv[i] & 
                                   pair == sibs_counts_with_ages$pair[i])
    for (t in cosmic_types_pyr){
        sibs_counts_with_ages[i,t] = sum(sib_sub$pyr == t)
    }
}

trio_counts = fread("./relatedness_dec20/trio_counts.txt", data.table = FALSE)
# add 7-type to trio counts 
trio_counts[cosmic_types_pyr] = NA
for (i in 1:nrow(trio_counts)){
    trio_sub = trio_difs %>% filter(par1 == trio_counts$par1[i] & 
                                   par2 == trio_counts$par2[i] & 
                                   off == trio_counts$off[i])
    for (t in cosmic_types_pyr){
        trio_counts[i,t] = sum(trio_sub$pyr == t)
    }
    
}
trio_counts$idv = trio_counts$off 
trio_counts$count = trio_counts$num_difs
trio_counts$denom = 2*2294489308
trio_counts$source = "trio"
trio_counts$pair = paste(trio_counts$par1, trio_counts$par2, trio_counts$off, sep = "_")
trio_counts = trio_counts %>% select(names(sibs_counts_with_ages))

aou_counts = rbind(sibs_counts_with_ages, trio_counts)

fwrite(aou_counts, "./aou_counts.txt", sep = "\t", 
      row.names= FALSE, quote = FALSE)

In [ ]:
# read in ukb counts 
ukb_sib = fread("./ukb/ukb_sibs_dnms_parental_ages.tsv", sep = "\t")
ukb_sib$pair = paste(ukb_sib$sib1, ukb_sib$sib2, sep = "_")
# merge with ibd2 
ibd2 = fread("./ukb_ibd2_giab_removed.tsv")
ukb_sib = merge(ukb_sib, ibd2, by = "pair")

ukb_sib = data.frame(idv = c(ukb_sib$sib1, ukb_sib$sib2), count = c(ukb_sib$sib1_dnms, ukb_sib$sib2_dnms),
                    denom = ukb_sib$length_filter, mother_age = c(ukb_sib$sib1_mother_age, ukb_sib$sib2_mother_age),
                    father_age = c(ukb_sib$sib1_father_age, ukb_sib$sib2_father_age), 
                    source = "sib", pair = ukb_sib$pair)
ukb_sib$denom = 2 * ukb_sib$denom

ukb_sib = ukb_sib %>%
  mutate(count_json = str_replace_all(count, "'", '"')) %>%
  mutate(parsed = map(count_json, fromJSON)) %>%
  unnest_wider(parsed) %>%
  select(-count_json)

ukb_sib = ukb_sib %>%
  mutate(count = rowSums(across(all_of(cosmic_types_pyr)), na.rm = TRUE))
ukb_sib = ukb_sib %>% select(names(aou_counts))

ukb_trio = fread("./ukb/ukb_trios_dnms_parental_ages.tsv", sep = "\t")
ukb_trio = data.frame(idv = ukb_trio$ind, count = ukb_trio$ind_dnms, denom = 2*2294489308,
                     mother_age = ukb_trio$ind_mother_age, father_age = ukb_trio$ind_father_age, 
                     source = "trio", pair = ukb_trio$ind)
ukb_trio = ukb_trio %>%
  mutate(count_json = str_replace_all(count, "'", '"')) %>%
  mutate(parsed = map(count_json, fromJSON)) %>%
  unnest_wider(parsed) %>%
  select(-count_json)

ukb_trio = ukb_trio %>%
  mutate(count = rowSums(across(all_of(cosmic_types_pyr)), na.rm = TRUE))
ukb_trio = ukb_trio %>% select(names(aou_counts))

ukb_counts = rbind(ukb_sib, ukb_trio)
aou_counts$dataset = "aou"
ukb_counts$dataset = "ukb"

In [ ]:
# read in ukb pcs
ukb_pcs = fread("./pca/ukb_pcs_biallelic.txt") %>% select(s, paste("pc", 1:6, sep = ""))
aou_pcs = fread("./pca/aou_pcs_biallelic.txt") %>% select(s, paste("pc", 1:6, sep = ""))
ukb_pcs$s = sapply(ukb_pcs$s, function(x) strsplit(x, "_")[[1]][1])
                   

In [ ]:
# merge counts with pcs
aou_counts = merge(aou_counts, aou_pcs, by.x = "idv", by.y = "s")
ukb_counts = merge(ukb_counts, ukb_pcs, by.x = "idv", by.y = "s")
all_counts = rbind(aou_counts, ukb_counts)
all_counts = all_counts[order(all_counts$idv, -all_counts$denom), ]
all_counts = all_counts[!duplicated(all_counts$idv), ]

In [ ]:
# assign ancestry

centers = kmeans_result$centers          # 8 x p
train_mat = as.matrix(meta_kmeans)       # same data used in kmeans
train_cl  = kmeans_result$cluster        # cluster assignment for training data

# squared euclidean distance to own centroid for training points
train_d2 = rowSums((train_mat - centers[train_cl, ])^2)
cluster_p95_d2 = tapply(train_d2, train_cl, quantile, probs = 0.95)

# new data
new_mat = as.matrix(all_counts[, paste("pc", 1:6, sep = "")])

# distance from each new point to each centroid
dist_mat = sapply(1:nrow(centers), function(k) {
  rowSums((t(t(new_mat) - centers[k, ]))^2)   # squared distance
})

# assigned cluster = nearest centroid
new_cluster = max.col(-dist_mat)

# squared distance to assigned centroid
new_d2 = dist_mat[cbind(seq_len(nrow(new_mat)), new_cluster)]

all_counts$cluster      = new_cluster
all_counts$dist2        = new_d2

# z-score: how many sds away from the typical distance in that cluster

all_counts$good_fit_95  = new_d2 <= cluster_p95_d2[new_cluster]

# keep only neatly fitting points
all_counts_clean = all_counts[which(all_counts$good_fit_95), ]

In [ ]:
fwrite(all_counts_clean, "./all_counts_clean.txt", sep = "\t", 
      row.names= FALSE, quote = FALSE)
fwrite(all_counts, "./all_counts.txt", sep = "\t", 
      row.names= FALSE, quote = FALSE)
genome = fread("./hg38.chrom.sizes")
genome = genome %>% filter(V1 %in% paste("chr", 1:22, sep = ""))
genome = sum(genome$V2)

anc_levels = c("EUR3", "EUR2", "EUR1", "AMR2", "MID", "SAS", "AMR1","EAS", "OCE", "AFR")

In [ ]:
# mutation rate x ancestry using trios only 

df = all_counts_clean %>% filter(!is.na(mother_age) & !is.na(father_age)) %>% filter(source == "trio") 
counts = table(df$cluster)
keep = rownames(counts[which(counts > 50)])

df = df %>% filter(cluster %in% keep) %>% 
  mutate(
    mut_count = count,
    ancestry = factor(cluster),
    batch    = factor(dataset),
    callable_genome = denom,
  ) %>%
  filter(callable_genome > 0)
df$ancestry = relevel(factor(df$ancestry), ref = "9")

df = df[is.finite(df$mut_count) &
         is.finite(df$father_age) &
         is.finite(df$mother_age) &
         is.finite(df$callable_genome) &
         df$callable_genome > 0, ]
table(df$cluster)
# compare to poisson
fit_qp = glm(
  mut_count ~ father_age + mother_age + batch + ancestry + offset(log(callable_genome)),
  data = df, family = quasipoisson(link = "log")
)

# create prediction data: vary ancestry, hold others at mean/mode
newdata = expand.grid(
  ancestry = unique(df$ancestry),
  father_age = mean(df$father_age),
  mother_age = mean(df$mother_age),
  batch = 'ukb',
  callable_genome = 1
)

# predict on response scale with se
preds = predict(fit_qp, newdata = newdata, type = "link", se.fit = TRUE)

# ci on log scale
newdata$lower_link = preds$fit - 1.96 * preds$se.fit
newdata$upper_link = preds$fit + 1.96 * preds$se.fit

# transform to response scale
newdata$expected = exp(preds$fit)
newdata$lower    = exp(newdata$lower_link)
newdata$upper    = exp(newdata$upper_link)
# plot
base_theme = theme_classic(base_size = 16) +
  theme(
    axis.text.x        = element_text(angle = 45, hjust = 1, size = 14),
    axis.text.y        = element_text(size = 14),
    axis.title         = element_text(size = 15),
    axis.title.y       = element_text(size = 15, margin = margin(r = 8)),
    panel.grid.major.y = element_line(color = "grey88", linewidth = 0.4)
  )

rate_df = newdata %>%
  mutate(ancestry = factor(cluster_to_anc[as.character(ancestry)], levels = anc_levels))

p_rate_cluster_trios = ggplot(rate_df, aes(ancestry, expected, color = ancestry)) +
  geom_point(size = 5) +
  geom_errorbar(aes(ymin = lower, ymax = upper), width = 0.4, linewidth = 2) +
  scale_color_manual(values = ancestry_colors, guide = "none") +
  labs(x = NULL, y = "DNM rate", title = "") +
  base_theme
mean(df$father_age)
mean(df$mother_age)
summary(fit_qp)
drop1(fit_qp, test = 'F')

In [ ]:
# 2 (afr) vs. non-2 (non-afr) using all samples with paternal age 

df = all_counts_clean %>% filter(!is.na(father_age)) 
counts = table(df$cluster)
keep = rownames(counts[which(counts > 50)])
# set 2 as ref and non-african as rest 
df$cluster[which(df$cluster != 2)] = "non_2"
keep = c("2", "non_2")

df = df %>% filter(cluster %in% keep) %>% 
  mutate(
    mut_count = count,
    ancestry = factor(cluster),
    batch    = factor(dataset),
    source = factor(source),
    callable_genome = denom,
  ) %>%
  filter(callable_genome > 0)

df$ancestry = relevel(factor(df$ancestry), ref = "non_2")

df = df[is.finite(df$mut_count) &
         is.finite(df$father_age) &
         is.finite(df$callable_genome) &
         df$callable_genome > 0, ]
table(df$cluster)
# compare to poisson
fit_qp = glm(
  mut_count ~ father_age + batch + ancestry + source + offset(log(callable_genome)),
  data = df, family = quasipoisson(link = "log")
)

# create prediction data: vary ancestry, hold others at mean/mode
newdata = expand.grid(
  ancestry = unique(df$ancestry),
  father_age = mean(df$father_age),
  mother_age = mean(df$mother_age),
  batch = 'ukb',
    source = 'sib',
  callable_genome = 1
)

# predict on response scale with se
preds = predict(fit_qp, newdata = newdata, type = "link", se.fit = TRUE)

# ci on log scale
newdata$lower_link = preds$fit - 1.96 * preds$se.fit
newdata$upper_link = preds$fit + 1.96 * preds$se.fit

# transform to response scale
newdata$expected = exp(preds$fit)
newdata$lower    = exp(newdata$lower_link)
newdata$upper    = exp(newdata$upper_link)

cluster_to_anc_2 = c("2" = "AFR", "non_2" = "non-AFR")

ancestry_colors_2 = c("AFR" = "#56B4E9", "non-AFR" = "gray")

rate_df = newdata %>%
  mutate(ancestry = factor(cluster_to_anc_2[as.character(ancestry)]))

p_rate_cluster_paternal = ggplot(rate_df, aes(ancestry, expected, color = ancestry)) +
  geom_point(size = 5) +
  geom_errorbar(aes(ymin = lower, ymax = upper), width = 0.4, linewidth = 2) +
  scale_color_manual(values = ancestry_colors_2, guide = "none") +
  labs(x = NULL, y = "DNM rate", title = "") +
  base_theme
mean(df$father_age)
summary(fit_qp)
drop1(fit_qp, test = 'F')
p_rate_cluster_paternal

In [ ]:
# mutation rate x ancestry using all samples with paternal age 

df = all_counts_clean %>% filter(!is.na(father_age)) 
counts = table(df$cluster)
keep = rownames(counts[which(counts > 50)])

df = df %>% filter(cluster %in% keep) %>% 
  mutate(
    mut_count = count,
    ancestry = factor(cluster),
    batch    = factor(dataset),
    source = factor(source),
    callable_genome = denom,
  ) %>%
  filter(callable_genome > 0)

df$ancestry = relevel(factor(df$ancestry), ref = "9")

df = df[is.finite(df$mut_count) &
         is.finite(df$father_age) &
         is.finite(df$callable_genome) &
         df$callable_genome > 0, ]
table(df$cluster)
# compare to poisson
fit_qp = glm(
  mut_count ~ father_age + batch + ancestry + source + 
    offset(log(callable_genome)),
  data = df, family = quasipoisson(link = "log")
)

# create prediction data: vary ancestry, hold others at mean/mode
newdata = expand.grid(
  ancestry = unique(df$ancestry),
  father_age = mean(df$father_age),
  mother_age = mean(df$mother_age),
  batch = 'ukb',
    source = 'sib',
  callable_genome = 1
)

# predict on response scale with se
preds = predict(fit_qp, newdata = newdata, type = "link", se.fit = TRUE)

# ci on log scale
newdata$lower_link = preds$fit - 1.96 * preds$se.fit
newdata$upper_link = preds$fit + 1.96 * preds$se.fit

# transform to response scale
newdata$expected = exp(preds$fit)
newdata$lower    = exp(newdata$lower_link)
newdata$upper    = exp(newdata$upper_link)

rate_df = newdata %>%
  mutate(ancestry = factor(cluster_to_anc[as.character(ancestry)], levels = anc_levels))

p_rate_cluster_sib = ggplot(rate_df, aes(ancestry, expected, color = ancestry)) +
  geom_point(size = 5) +
  geom_errorbar(aes(ymin = lower, ymax = upper), width = 0.4, linewidth = 2) +
  scale_color_manual(values = ancestry_colors, guide = "none") +
  labs(x = NULL, y = "DNM rate", title = "DNM rate adjusted for paternal age using trio+sibling data") +
  base_theme

In [ ]:
# mutation proportion x ancestry using all samples with paternal age 

df = all_counts_clean  %>% filter(!is.na(father_age))
counts = table(df$cluster)
keep = rownames(counts[which(counts > 60)])

df = df %>% filter(cluster %in% keep) %>% 
  mutate(
    mut_count = count,
    mut_rate = count/denom,
    ancestry = factor(cluster),
    batch    = factor(dataset),
    source = factor(source),
    callable_genome = denom,
  ) %>%
  filter(callable_genome > 0 & 
        mut_count > 0)

df$ancestry = relevel(factor(df$ancestry), ref = "9")

table(df$cluster)

## 0) make sure these are factors (important for emmeans + contrasts)
df = df %>%
  mutate(
    ancestry = factor(ancestry),
    batch    = factor(batch),
    source   = factor(source)
  )

mut_types = cosmic_types_pyr

fit_one = function(y) {
  f = as.formula(sprintf("cbind(`%s`,mut_count-`%s`)  ~ source + father_age + batch + ancestry",y, y))
  glm(f, family = binomial(link = "log"), data = df)
}

fits = set_names(map(mut_types, fit_one), mut_types)


overall_terms = imap_dfr(fits, function(fit, y) {

  tab = drop1(fit, test = "Chisq")              # overall term tests for quasi-glm
  tab = as.data.frame(tab)
  tab$term = rownames(tab)
  tab$mut_type = y

  tab
}) %>%
  filter(term != "<none>") %>%
  rename(
    df       = Df,
    p_value  = `Pr(>Chi)`
  ) %>%
  relocate(mut_type, term, df, p_value) %>%
  mutate(
    p_adj_global = p.adjust(p_value, method = "BH")
  ) %>%
  ungroup()

overall_wide = overall_terms %>%
  select(mut_type, term, p_value, p_adj_global) %>%
  pivot_wider(
    names_from  = term,
    values_from = c(p_value, p_adj_global),
    names_sep   = "__"
  ) %>%
  arrange(p_adj_global__ancestry)

overall_wide

library(dplyr)
library(xtable)

fmt_p = function(p) {
  ifelse(is.na(p), "", formatC(p, format = "e", digits = 2))
}

overall_terms_xt = overall_terms %>% select(-Deviance, -AIC) %>%
  mutate(
    df = as.integer(df),
    p_value  = fmt_p(p_value),
    p_adj_global       = fmt_p(p_adj_global),
  ) %>%
  rename(
    Mutation = mut_type,
    Term = term,
    Df = df,
    p = p_value,
    BH_global = p_adj_global,
  ) %>%
  arrange(Mutation, Term)

xt = xtable(
  overall_terms_xt,
  caption = "Overall binomial GLM term tests across mutation types.",
  label   = "tab:overall_term_tests"
)

print(
  xt,
  include.rownames = FALSE,
  booktabs = TRUE,
  caption.placement = "top",
  comment = FALSE
)

df = df %>%
  mutate(
    ancestry = factor(ancestry),
    batch    = factor(batch),
    source   = factor(source)
  )

mut_types = cosmic_types_pyr

ref_vals = list(
  ancestry  = levels(df$ancestry),         # emmeans will iterate these
  mut_rate  = mean(df$mut_rate, na.rm=TRUE),
  batch     = "ukb",
  source    = "sib",
  mut_count = 1,
    father_age = mean(df$father_age)
)

pairwise_tbl = imap_dfr(fits, \(fit, y) {
  emm = emmeans(fit, ~ ancestry, weights = "proportional")

  as.data.frame(
    summary(
      pairs(emm, adjust = "none"),
      infer = c(TRUE, TRUE),
      type = "response"
    )
  ) %>%
    mutate(mut_type = y, .before = 1) %>%
    separate(contrast, into = c("anc1", "anc2"), sep = "/", remove = FALSE)
})

pairwise_tbl = pairwise_tbl %>% filter(mut_type %in% c("C>T")) %>%
  mutate(p_adj_global = p.adjust(p.value, method = "fdr"))

sig_pairs_global = pairwise_tbl %>%
  filter(p_adj_global < 0.05) %>%
  arrange(mut_type, p_adj_global)


ref_vals = list(
  ancestry  = levels(df$ancestry),         # emmeans will iterate these
  mut_rate  = mean(df$mut_rate, na.rm=TRUE),
  batch     = "ukb",
  source    = "sib",
  mut_count = 1,
    mother_age = mean(df$mother_age),
    father_age = mean(df$father_age)
)

emm_plot_df = imap_dfr(fits, function(fit, y) {
  emm_resp = emmeans(fit, ~ ancestry, at = ref_vals, type = "response")
  as.data.frame(confint(emm_resp)) %>%
    mutate(mut_type = y)
})

# correct p-values 
for (i in unique(overall_terms$term)){
    overall_terms$p_adj_global[which(overall_terms$term == i)] = p.adjust(overall_terms$p_value[which(overall_terms$term == i)], method = "fdr")
}

overall_wide = overall_terms %>%
  select(mut_type, term, p_value, p_adj_global) %>%
  pivot_wider(
    names_from  = term,
    values_from = c(p_value, p_adj_global),
    names_sep   = "__"
  ) %>%
  arrange(p_adj_global__ancestry)

overall_wide

library(dplyr)
library(xtable)


overall_terms_xt = overall_terms %>% select(-Deviance, -AIC) %>%
  mutate(
    df = as.integer(df),
    p_value  = fmt_p(p_value),
    p_adj_global       = fmt_p(p_adj_global),
      term = ifelse(term == "batch", "biobank", term)
  ) %>%
  rename(
    Mutation = mut_type,
    Term = term,
    Df = df,
    p = p_value,
    BH_global = p_adj_global,
  ) %>%
  arrange(Term, Mutation)

xt = xtable(
  overall_terms_xt,
  caption = "Overall binomial GLM term tests across mutation types.",
  label   = "tab:overall_term_tests"
)
print(
  xt,
  include.rownames = FALSE,
  booktabs = TRUE,
  caption.placement = "top",
  comment = FALSE
)

In [ ]:
# mutation proportion x ancestry using trios only 

df = all_counts_clean  %>% filter(source == "trio")
counts = table(df$cluster)
keep = rownames(counts[which(counts > 60)])

df = df %>% filter(cluster %in% keep) %>% 
  mutate(
    mut_count = count,
    mut_rate = count/denom,
    ancestry = factor(cluster),
    batch    = factor(dataset),
    source = factor(source),
    callable_genome = denom,
  ) %>%
  filter(callable_genome > 0 & 
        mut_count > 0)

df$ancestry = relevel(factor(df$ancestry), ref = "9")

table(df$cluster)

df = df %>%
  mutate(
    ancestry = factor(ancestry),
    batch    = factor(batch),
    source   = factor(source)
  )

mut_types = cosmic_types_pyr

fit_one = function(y) {
  f = as.formula(sprintf("cbind(`%s`,mut_count-`%s`)  ~ mother_age + father_age + batch + ancestry",y, y))
  glm(f, family = binomial(link = "log"), data = df)
}

fits = set_names(map(mut_types, fit_one), mut_types)


overall_terms = imap_dfr(fits, function(fit, y) {

  tab = drop1(fit, test = "Chisq")              # overall term tests for quasi-glm
  tab = as.data.frame(tab)
  tab$term = rownames(tab)
  tab$mut_type = y

  tab
}) %>%
  filter(term != "<none>") %>%
  rename(
    df       = Df,
    p_value  = `Pr(>Chi)`
  ) %>%
  relocate(mut_type, term, df, p_value) %>%
  mutate(
    p_adj_global = p.adjust(p_value, method = "BH")
  ) %>%
  ungroup()

overall_wide = overall_terms %>%
  select(mut_type, term, p_value, p_adj_global) %>%
  pivot_wider(
    names_from  = term,
    values_from = c(p_value, p_adj_global),
    names_sep   = "__"
  ) %>%
  arrange(p_adj_global__ancestry)

overall_wide

library(dplyr)
library(xtable)

fmt_p = function(p) {
  ifelse(is.na(p), "", formatC(p, format = "e", digits = 2))
}

overall_terms_xt = overall_terms %>% select(-Deviance, -AIC) %>%
  mutate(
    df = as.integer(df),
    p_value  = fmt_p(p_value),
    p_adj_global       = fmt_p(p_adj_global),
  ) %>%
  rename(
    Mutation = mut_type,
    Term = term,
    Df = df,
    p = p_value,
    BH_global = p_adj_global,
  ) %>%
  arrange(Mutation, Term)

xt = xtable(
  overall_terms_xt,
  caption = "Overall binomial GLM term tests across mutation types.",
  label   = "tab:overall_term_tests"
)

print(
  xt,
  include.rownames = FALSE,
  booktabs = TRUE,
  caption.placement = "top",
  comment = FALSE
)

df = df %>%
  mutate(
    ancestry = factor(ancestry),
    batch    = factor(batch),
    source   = factor(source)
  )

mut_types = cosmic_types_pyr

ref_vals = list(
  ancestry  = levels(df$ancestry),         # emmeans will iterate these
  mut_rate  = mean(df$mut_rate, na.rm=TRUE),
  batch     = "ukb",
  source    = "sib",
  mut_count = 1,
    father_age = mean(df$father_age)
)

pairwise_tbl = imap_dfr(fits, \(fit, y) {
  emm = emmeans(fit, ~ ancestry, weights = "proportional")

  as.data.frame(
    summary(
      pairs(emm, adjust = "none"),
      infer = c(TRUE, TRUE),
      type = "response"
    )
  ) %>%
    mutate(mut_type = y, .before = 1) %>%
    separate(contrast, into = c("anc1", "anc2"), sep = "/", remove = FALSE)
})

pairwise_tbl = pairwise_tbl %>% filter(mut_type %in% c("C>T")) %>%
  mutate(p_adj_global = p.adjust(p.value, method = "fdr"))

sig_pairs_global = pairwise_tbl %>%
  filter(p_adj_global < 0.05) %>%
  arrange(mut_type, p_adj_global)


ref_vals = list(
  ancestry  = levels(df$ancestry),         # emmeans will iterate these
  mut_rate  = mean(df$mut_rate, na.rm=TRUE),
  batch     = "ukb",
  source    = "sib",
  mut_count = 1,
    mother_age = mean(df$mother_age),
    father_age = mean(df$father_age)
)

emm_plot_df = imap_dfr(fits, function(fit, y) {
  emm_resp = emmeans(fit, ~ ancestry, at = ref_vals, type = "response")
  as.data.frame(confint(emm_resp)) %>%
    mutate(mut_type = y)
})

In [ ]:

# ── panel a: pca ──────────────────────────────────────────────────────────────
pca_df = meta %>%
  mutate(ancestry = factor(cluster_to_anc[as.character(cluster)], levels = anc_levels))

p_pca = ggplot(pca_df, aes(pc1, pc2, color = ancestry)) +
  geom_point(size = 2, alpha = 1) +
  scale_color_manual(
    values = ancestry_colors,
    labels = legend_labels,
    name   = NULL,
    drop   = FALSE
  ) +
  labs(x = "PC1", y = "PC2") +
  guides(color = guide_legend(
    override.aes = list(size = 5),
    ncol = 1
  )) +
  theme_classic(base_size = 16) +
  theme(
    legend.position      = c(0.99, 0.99),
    legend.justification = c(1, 1),
    legend.background    = element_rect(
      fill = alpha("white", 0.9), color = "grey80", linewidth = 0.3
    ),
    legend.key.size  = unit(0.6, "cm"),
    legend.text      = element_text(size = 12),
    legend.spacing.y = unit(0.1, "cm"),
    panel.grid.major = element_line(color = "grey88", linewidth = 0.4)
  )

# ── panels b/c: histograms ────────────────────────────────────────────────────
hist_df = all_counts_clean %>%
  mutate(ancestry = factor(cluster_to_anc[as.character(cluster)], levels = anc_levels)) %>%
  filter(!is.na(ancestry)) %>%
  count(dataset, ancestry, .drop = FALSE)

make_hist = function(ds, show_x = FALSE) {
  hist_df %>%
    filter(dataset == ds) %>%
    ggplot(aes(ancestry, n, fill = ancestry)) +
    geom_col(width = 0.72) +
    scale_fill_manual(values = ancestry_colors, drop = FALSE, guide = "none") +
    scale_y_continuous(
      expand = expansion(mult = c(0, 0.08)),
      labels = scales::label_comma()
    ) +
    labs(
      x     = NULL,
      y     = "Count",
      title = if (ds == "aou") "AoU" else toupper(ds)
    ) +
    base_theme +
    theme(
      plot.title   = element_text(hjust = 0.5, size = 15, face = "bold"),
      axis.text.x  = if (show_x)
        element_text(angle = 45, hjust = 1, size = 13)
      else
        element_blank(),
      axis.ticks.x = if (show_x) element_line() else element_blank()
    )
}

p_ukb = make_hist("ukb", show_x = FALSE)
p_aou = make_hist("aou", show_x = TRUE)

# ── panel d: mutation rates ───────────────────────────────────────────────────
rate_df = newdata %>%
  mutate(ancestry = factor(cluster_to_anc[as.character(ancestry)], levels = anc_levels))

p_rate = ggplot(rate_df, aes(ancestry, expected, color = ancestry)) +
  geom_point(size = 5) +
  geom_errorbar(aes(ymin = lower, ymax = upper), width = 0.4, linewidth = 1) +
  scale_color_manual(values = ancestry_colors, guide = "none") +
  labs(x = NULL, y = "DNM rate") +
  base_theme

# ── panels e/f: c>t and t>c separately ───────────────────────────────────────
prop_df = emm_plot_df %>%
  mutate(
    ancestry = factor(
      cluster_to_anc[gsub("ancestry", "", as.character(ancestry))],
      levels = anc_levels
    )
  ) %>%
  filter(!is.na(ancestry))

sig_df = pairwise_tbl %>%
  filter(mut_type %in% c("C>T", "T>C"), p_adj_global < 0.05) %>%
  mutate(
    anc1 = factor(
      cluster_to_anc[as.numeric(gsub("ancestry", "", as.character(anc1)))],
      levels = anc_levels
    ),
    anc2 = factor(
      cluster_to_anc[as.numeric(gsub("ancestry", "", as.character(anc2)))],
      levels = anc_levels
    ),
    label = case_when(
      p_adj_global < 0.001 ~ "***",
      p_adj_global < 0.01  ~ "**",
      TRUE                  ~ "*"
    )
  ) %>%
  filter(!is.na(anc1), !is.na(anc2))

# function to build one proportion panel
make_prop_plot = function(mut, show_x = FALSE) {
  df   = prop_df %>% filter(mut_type == mut)
  sigs = sig_df  %>% filter(mut_type == mut)

  y_max_val = max(df$asymp.UCL, na.rm = TRUE)

  if (nrow(sigs) > 0) {
    sigs = sigs %>%
  mutate(
    xmin      = as.character(anc1),
    xmax      = as.character(anc2),
    xmin_int  = as.integer(factor(anc1, levels = anc_levels)),
    xmax_int  = as.integer(factor(anc2, levels = anc_levels)),
  )
      
   sigs = sigs   %>% 
  arrange(anc1, desc = TRUE) %>%
  mutate(
    rank       = row_number(),
    y_position = y_max_val * (1.02 + (rank - 1) * 0.01),
    y_tip      = y_position
  )

  }

  p = ggplot(df, aes(ancestry, prob, color = ancestry)) +
    geom_point(size = 5) +
    geom_errorbar(
      aes(ymin = asymp.LCL, ymax = asymp.UCL),
      width = 0.3, linewidth = 1
    ) +
    scale_color_manual(values = ancestry_colors, guide = "none") +
    scale_y_continuous(n.breaks = 8) +
    labs(x = NULL, y = "Proportion", title = mut) +
    base_theme +
    theme(
      plot.title   = element_text(face = "bold", size = 15, hjust = 0.5),
      axis.text.x  = if (show_x)
        element_text(angle = 45, hjust = 1, size = 13)
      else
        element_blank(),
      axis.ticks.x = if (show_x) element_line() else element_blank()
    ) +
    coord_cartesian(clip = "off")

        
  if (nrow(sigs) > 0) {

      
    p = p +
      geom_segment(
        data        = sigs,
        aes(x = xmin, xend = xmax, y = y_position, yend = y_position),
        inherit.aes = FALSE, color = "grey30", linewidth = 1
      ) 
  }

  p
}

p_ct = make_prop_plot("C>T", show_x = FALSE)
p_tc = make_prop_plot("T>C", show_x = TRUE)
p_ta = make_prop_plot("T>A", show_x = TRUE)
p_tg = make_prop_plot("T>G", show_x = TRUE)
p_cg = make_prop_plot("C>G", show_x = TRUE)
p_cpg = make_prop_plot("CpG>TpG", show_x = TRUE)
p_ca = make_prop_plot("C>A", show_x = TRUE)
        

# ── assemble ──────────────────────────────────────────────────────────────────
tag_theme = theme(plot.tag = element_text(size = 18, face = "bold"), axis.text.y = element_text(hjust = 1))

hist_col = (p_ukb + tag_theme)/ (p_aou + tag_theme)
spec_col  = (p_ct + tag_theme) / (p_tc + tag_theme) 

figure = (
  ((p_pca + tag_theme) |
  (hist_col + tag_theme)) /
  (((p_rate + tag_theme) | (p_ct + tag_theme)) / ((p_tc + tag_theme) | (p_cg + tag_theme)))
) +
  plot_layout(widths = c(1, 1), heights = c(1,1.2)) +
  plot_annotation(tag_levels = "a")

figure 



In [ ]:
library(cowplot)

top_row = plot_grid(
  p_ct + tag_theme, p_ca + tag_theme, p_cg + tag_theme, p_cpg + tag_theme,
  nrow = 1, align = "hv", axis = "tblr"
)

bottom_row = plot_grid(
  p_tc + tag_theme, p_ta + tag_theme, p_tg + tag_theme, NULL,
  nrow = 1, align = "hv", axis = "tblr"
)

figure = plot_grid(top_row, bottom_row, nrow = 2)

In [ ]:
ukb = fread("./ukb/ukb_96_type_dnms_feb11.csv", sep = ",")
ukb$off_pc = sapply(ukb$FAM, function(x) sample(strsplit(x, "_")[[1]], 1))
ukb_pcs = fread("./pca/ukb_pcs_biallelic_jan25.txt")
ukb_pcs = ukb_pcs %>% select(s, paste("pc", 1:6, sep = ""))
ukb_pcs$s = sapply(ukb_pcs$s, function(x) strsplit(x, "_")[[1]][1])
ukb_pc_added = merge(ukb, ukb_pcs, by.x = "off_pc", by.y = "s")
fwrite(ukb_pc_added, "./ukb/ukb_96_type_dnms_feb11_pc_added.csv", sep = ",",
      row.names = FALSE, quote = FALSE)

In [ ]:
aou_96 = fread("./gts/aou_gt_lof_indel_snps_ages_added.txt")
extract = names(ukb_pc_added)
aou_96 = aou_96[,..extract]
ukb_pc_added$dataset = "ukb"
aou_96$dataset = "aou"
all_sib_trio_pcs = rbind(ukb_pc_added, aou_96)

In [ ]:
centers = kmeans_result$centers          # 8 x p
train_mat = as.matrix(meta_kmeans)       # same data used in kmeans
train_cl  = kmeans_result$cluster        # cluster assignment for training data

# squared euclidean distance to own centroid for training points
train_d2 = rowSums((train_mat - centers[train_cl, ])^2)
cluster_p95_d2 = tapply(train_d2, train_cl, quantile, probs = 0.95)

In [ ]:
# new data
new_mat = as.matrix(all_sib_trio_pcs[, paste("pc", 1:6, sep = "")])

# distance from each new point to each centroid
dist_mat = sapply(1:nrow(centers), function(k) {
  rowSums((t(t(new_mat) - centers[k, ]))^2)   # squared distance
})

# assigned cluster = nearest centroid
new_cluster = max.col(-dist_mat)

# squared distance to assigned centroid
new_d2 = dist_mat[cbind(seq_len(nrow(new_mat)), new_cluster)]

all_sib_trio_pcs$cluster      = new_cluster
all_sib_trio_pcs$dist2        = new_d2

# z-score: how many sds away from the typical distance in that cluster

all_sib_trio_pcs$good_fit_95  = new_d2 <= cluster_p95_d2[new_cluster]

# keep only neatly fitting points
all_96_clean = all_sib_trio_pcs[which(all_sib_trio_pcs$good_fit_95), ]

all_96 = all_sib_trio_pcs
fwrite(all_96, "./aou_ukb_families_COSMIC_96_all_samples.txt", sep = "\t",
      quote=FALSE, row.names=FALSE)
fwrite(all_96_clean , "./aou_ukb_families_COSMIC_96_passing_samples.txt", sep = "\t",
      quote=FALSE, row.names=FALSE)


In [ ]:
family = fread("./aou_ukb_families_COSMIC_96_passing_samples.txt")
family$tot_muts = rowSums(family[,..cosmic_96])
counts = family %>% filter(!is.na(father_age)) %>%
  group_by(cluster, dataset) %>%
  summarise(tot_muts = sum(tot_muts), .groups = "drop") %>% mutate(cluster = cluster_to_anc[cluster],
                                                                  dataset = c("aou" = "AoU", "ukb" = "UKB")[dataset])
names(counts) = c("Ancestry", "Dataset",  "DNMs")

tot = family %>% filter(!is.na(father_age)) %>%
  group_by(cluster) %>%
  summarise(DNMs = sum(tot_muts), .groups = "drop") %>% mutate(Ancestry = cluster_to_anc[cluster])
tot$Dataset = "Combined"
tot = tot %>% select(Ancestry, Dataset, DNMs)
counts = rbind(counts, tot)

# numbers from garcia et al. 
garcia_trios = c("AFR" = 198, "AMR" = 215, "EAS" = 53, "EUR" = 8104, "SAS" = 1250)
garcia_ave = c("AFR" = 66.71, "AMR" = 63.99, "EAS" = 66, "EUR" = 64.20, "SAS" = 64.66 )
garcia_dnms = garcia_ave*garcia_trios
garcia = data.frame(Ancestry = names(garcia_trios),
                   Dataset = "Garcia Salinas et al. 2025",
                   DNMs = garcia_dnms)

counts_tab = rbind(counts, garcia) %>% mutate(Dataset = 
                                         factor(Dataset, levels = c("AoU", "UKB", "Combined",
                                                                      "Garcia Salinas et al. 2025")),
                                             Ancestry = factor(Ancestry, levels = c("AFR", "AMR1", "AMR2", "AMR", 
                                                                                   "EAS", "EUR1", "EUR2", "EUR3", "EUR", "MID", "SAS"))) %>% arrange(Ancestry, Dataset)

In [ ]:
df_fmt = counts_tab %>%
  mutate(DNMs = formatC(as.numeric(DNMs), format = "d", big.mark = ","))

print(
  xtable(df_fmt, caption = "DNM counts by ancestry, dataset, and source"),
  include.rownames = FALSE,
  booktabs = TRUE
)

In [ ]:
all_96_clusters = list()
for (i in 1:10){
    all_96_clusters[[i]] = all_96 %>% filter(cluster == i)
}
all_96_clusters_aou_sib = list()
for (i in 1:10){
    all_96_clusters_aou_sib[[i]] = all_96 %>% filter(cluster == i & dataset == "aou" & source == "sib")
}
cosmic_96 = names(all_96)[2:97]
calculate_cosmic = function(table_96){
    res = data.frame(context = cosmic_96, count = NA)
    for (i in 1:nrow(res)){
        context_i = res$context[i]
        res$count[i] = sum(table_96[,..context_i])
    }
    res$percent = res$count/sum(res$count)*100
    return(res)
}
cos_sim = function(x, y) {
  sum(x * y) / (sqrt(sum(x^2)) * sqrt(sum(y^2)))
}
sig_pairs_global$anc1 = gsub(sig_pairs_global$anc1, pattern = "ancestry", replace = "")
sig_pairs_global$anc2 = gsub(sig_pairs_global$anc2, pattern = "ancestry", replace = "")
ct_compare = sig_pairs_global %>% filter(mut_type == "C>T")

compare_vectors = function(v1, v2){
    res = rep(NA, length(v1))
    tot1 = sum(v1)
    tot2 = sum(v2)
    for (i in 1:length(v1)){
        d =  matrix(c(v1[i], tot1-v1[i], v2[i], tot2-v2[i]), nrow = 2, byrow=TRUE)
        res[i] = chisq.test(d)$p.value
    }
    sorted = order(res, decreasing = FALSE)
    # for lowest p-value, the ordered p is = unordered p 
    ordered_res = rep(NA, length(v1))
    ordered_res[sorted[1]] = res[sorted[1]]
    for (i in 2:(length(sorted)-1)){ # for the rest
        tot1 = sum(v1[sorted[(i+1):length(sorted)]])# sum from i+1 to end 
        tot2 = sum(v2[sorted[(i+1):length(sorted)]])
        d =  matrix(c(v1[sorted[i]], tot1, v2[sorted[i]], tot2), nrow = 2, byrow=TRUE)
        ordered_res[sorted[i]] = chisq.test(d)$p.value
    }
     d =  matrix(c(v1[sorted[length(sorted)]], v1[sorted[(length(sorted)-1)]], v2[sorted[length(sorted)]], v2[sorted[(length(sorted)-1)]]), nrow = 2, byrow = TRUE)
     ordered_res[sorted[length(sorted)]] = chisq.test(d)$p.value  
    return(ordered_res)
}
compare_tables = function(table_a, table_b){
    names(table_a) = c("context", "count_a", "percent_a")
    names(table_b) = c("context", "count_b", "percent_b")
    merged = merge(table_a, table_b, by = "context")
    merged$p = compare_vectors(merged$count_a, merged$count_b)
    merged$percent_a_minus_b = merged$percent_a - merged$percent_b
    merged$log2_a_over_b = log2(merged$percent_a/merged$percent_b)
    return(merged)
}

cosmic_types = cosmic_96
all_ct = cosmic_types[which(substr(cosmic_types, 3, 5) == "C>T" & 
                           substr(cosmic_types, 7, 7) != "G")]


In [ ]:
# 96-type ancestry c>t comparisons using aou siblings 

ct_comparisons = data.frame(context = NA, count_a = NA, percent_a = NA, count_b = NA, percent_b = NA, 
                                    p = NA,
                                   percent_a_minus_b = NA, log2_a_over_b = NA, cluster_a = NA, cluster_b = NA)
for (i in 1:nrow(ct_compare)){
    a = as.numeric(ct_compare$anc1[i])
    b = as.numeric(ct_compare$anc2[i])
    a_96 = calculate_cosmic(all_96 %>% filter(cluster == a & dataset == "aou" & source == "sib")) 
    b_96 = calculate_cosmic(all_96 %>% filter(cluster == b & dataset == "aou" & source == "sib")) 
    r = compare_tables(a_96, b_96) %>% filter(context %in% all_ct)
    r$cluster_a = paste("aou_sib_ct", a, sep = "")
    r$cluster_b = paste("aou_sib_ct", b, sep = "")
    ct_comparisons = rbind(ct_comparisons, r)
}
ct_comparisons = ct_comparisons[2:nrow(ct_comparisons),]
ct_comparisons$test = "aou_sib_ct"
all_tests = ct_comparisons
all_tests_aou = all_tests %>% filter(test %in% c("aou_sib_ct"))
all_tests_aou$fdr = p.adjust(all_tests_aou$p, method = "fdr")
all_tests_aou %>% filter(fdr <= 0.05)

In [ ]:
# plot 96-type ancestry c>t comparisons using aou siblings 



cosmic_compare = function(df,t){
# df: columns `context` (like "t[c>t]g") and `percent` (0–100 or 0–1)

# 1. parse context string
df_parsed = df %>%
  tidyr::extract(
    context,
    into = c("left", "ref", "alt", "right"),
    regex = "([ACGT])\\[([ACGT])>([ACGT])\\]([ACGT])"
  ) %>%
  mutate(
    substitution = paste0(ref, ">", alt),
    trinuc       = paste0(left, ref, right)
  ) %>%
  mutate(
    substitution = factor(
      substitution,
      levels = c("C>A", "C>G", "C>T", "T>A", "T>C", "T>G")  # mutation-type order
    ),
    trinuc = factor(trinuc, levels = sort(unique(trinuc)))   # alphabetical a..t
  )

# if `percent` is a fraction 0–1, uncomment this:
# df_parsed = df_parsed %>% mutate(percent = percent * 100)

# 3. plot cosmic-style 96-bar signature
p = ggplot(df_parsed, aes(x = trinuc, y = percent)) +
  geom_col(fill = "white",
    width = 0.7,
    color = "black",linewidth = 0.1
  ) + geom_col(data = df_parsed %>% filter(fdr <= 0.05), aes(fill = substitution),
    width = 0.7,
    color = "black",linewidth = 0.1
  ) + 
  facet_grid(~ substitution, scales = "free_x", space = "free_x") +
  scale_fill_manual(
    values = c(
      "C>A" = "#4C72B0",
      "C>G" = "#000000",
      "C>T" = "#D62728",
      "T>A" = "#7F7F7F",
      "T>C" = "#2CA02C",
      "T>G" = "#F4A6C1"
    )
  ) +
  scale_y_continuous(expand = expansion(mult = c(0.05, 0.05))) +
  labs(x = "Trinucleotide context", y = "Percent") +
  theme_bw(base_size = 16) +
  theme(
    axis.text.x   = element_text(angle = 90, vjust = 0.5, hjust = 1, size = 9),
    panel.spacing = unit(0.5, "lines"),
    strip.background = element_blank(),
    strip.text = element_text(face = "bold", size = 14),
    legend.position = "none",
    panel.grid = element_blank()   # = remove all gridlines
  ) + labs(title = t)

    return(p)
}

cosmic = function(df,t){
# df: columns `context` (like "t[c>t]g") and `percent` (0–100 or 0–1)

# 1. parse context string
df_parsed = df %>%
  tidyr::extract(
    context,
    into = c("left", "ref", "alt", "right"),
    regex = "([ACGT])\\[([ACGT])>([ACGT])\\]([ACGT])"
  ) %>%
  mutate(
    substitution = paste0(ref, ">", alt),
    trinuc       = paste0(left, ref, right)
  ) %>%
  mutate(
    substitution = factor(
      substitution,
      levels = c("C>A", "C>G", "C>T", "T>A", "T>C", "T>G")  # mutation-type order
    ),
    trinuc = factor(trinuc, levels = sort(unique(trinuc)))   # alphabetical a..t
  )

# if `percent` is a fraction 0–1, uncomment this:
# df_parsed = df_parsed %>% mutate(percent = percent * 100)

# 3. plot cosmic-style 96-bar signature
p = ggplot(df_parsed, aes(x = trinuc, y = percent)) +
  geom_col(
    aes(fill = substitution),
    width = 0.7,
    color = "black",linewidth = 0.1
  ) +
  facet_grid(~ substitution, scales = "free_x", space = "free_x") +
  scale_fill_manual(
    values = c(
      "C>A" = "#4C72B0",
      "C>G" = "#000000",
      "C>T" = "#D62728",
      "T>A" = "#7F7F7F",
      "T>C" = "#2CA02C",
      "T>G" = "#F4A6C1"
    )
  ) +
  scale_y_continuous(expand = expansion(mult = c(0.05, 0.05))) + lims(y = c(0,6))+
  labs(x = "Trinucleotide context", y = "Percent") +
  theme_bw(base_size = 9) +
  theme(
    axis.text.x   = element_text(angle = 90, vjust = 0.5, hjust = 1, size = 9),
    panel.spacing = unit(0.5, "lines"),
    strip.background = element_blank(),
    strip.text = element_text(face = "bold", size = 9),
    legend.position = "none",
    panel.grid = element_blank()   # = remove all gridlines
  ) + labs(title = t) 

    return(p)
}

# align properly for plotting
ct_compare_2 = ct_compare 
ct_compare_2$anc1 = ct_compare$anc2
ct_compare_2$anc2 = ct_compare$anc1
ct_compare = rbind(ct_compare, ct_compare_2)
ct_compare = ct_compare %>% filter(anc1==4)
nrow(ct_compare)
ct_compare$pair = paste(ct_compare$anc1, ct_compare$anc2)
unique_comparisons = unique(sig_pairs_global %>% select(anc1, anc2))
unique_comparisons_2 = unique_comparisons 
unique_comparisons_2$anc1 = unique_comparisons$anc2
unique_comparisons_2$anc2 = unique_comparisons$anc1
unique_comparisons = rbind(unique_comparisons, unique_comparisons_2)
unique_comparisons = unique_comparisons %>% filter(anc1 == 4)
nrow(unique_comparisons)

In [ ]:
replace_cluster_with_name = function(cluster){
    names = c("EAS", "AFR", "MID", "AMR1", "EUR1", "AMR2", 
             "SAS", "EUR2", "EUR3", "OCE")
    for (i in 1:10){
        cluster = gsub(cluster, pattern = i, replace = names[i])
    }
    return(cluster)
}
ps = c()

unique_comparisons = data.frame(anc1 = c(4,4), anc2 = c(5,9))
for (i in 1:nrow(unique_comparisons)){
    a = unique_comparisons$anc1[i]
    b =  unique_comparisons$anc2[i]
    pair = paste(a, b)
    a_96 = calculate_cosmic(all_96 %>% filter(cluster == as.numeric(a) & dataset == "aou" & source == "sib")) 
    b_96 = calculate_cosmic(all_96 %>% filter(cluster == as.numeric(b) & dataset == "aou" & source == "sib"))
    r = compare_tables(a_96, b_96)
    input = data.frame(context = r$context, percent = r$percent_a_minus_b)
    input$fdr = 100
    input$fdr = ifelse(input$context %in% all_ct, 0, input$fdr)

    p = cosmic_compare(input, t = paste(replace_cluster_with_name(a), " - ", replace_cluster_with_name(b)))
    ps = c(ps, p)
}

fig = ggarrange(plotlist = ps, ncol = 2, nrow = 1)
final_figure = annotate_figure(
  fig,
  top = text_grob("Mutation spectrum comparisons using AoU siblings", color = "black", size = 20)
)
final_figure

In [ ]:
# sas vs. eur3 comparison at c[c>t]g
a_96 = calculate_cosmic(all_96 %>% filter(cluster == 9 & dataset == "ukb" & source == "sib")) 
b_96 = calculate_cosmic(all_96 %>% filter(cluster == 7 & dataset == "ukb" & source == "sib"))
r = compare_tables(a_96, b_96)
r %>% filter(context == "C[C>T]G")
r %>% filter(fdr <= 0.05)

In [ ]:
# eur vs. eas+afr comparison at t[c>t]c
a_96 = calculate_cosmic(all_96 %>% filter(cluster %in% c(5,9) & dataset == "aou" & source == "sib")) 
b_96 = calculate_cosmic(all_96 %>% filter((cluster %in% c(1,2)) & dataset == "aou" & source == "sib"))
r = compare_tables(a_96, b_96)
r$fdr = p.adjust(r$p)
r %>% filter(context == "T[C>T]C")
r %>% filter(fdr <= 0.05)